In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR

In [2]:
df=pd.read_csv('/kaggle/input/datasets/anubhashukla/time-series-data/timeSeries.csv',parse_dates=['Timestamp'])
df_matrix=df.pivot(index='Timestamp',columns='Node',values='Power')

/tmp/ipykernel_58/4129797265.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df=pd.read_csv('/kaggle/input/datasets/anubhashukla/time-series-data/timeSeries.csv',parse_dates=['Timestamp'])


In [5]:
np.random.seed(42)
noise=np.random.normal(0,1e-6,df_matrix.shape)
df_matrix_jittered=df_matrix+noise

In [6]:
split_idx=int(len(df_matrix_jittered)*0.8)
train=df_matrix_jittered.iloc[:split_idx]
test=df_matrix_jittered.iloc[split_idx:]

In [7]:
model=VAR(train)
results=model.fit(maxlags=3,ic='aic')
print(f'Optimal historical lag window: {results.k_ar}')

Optimal historical lag window: 1


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:559: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  _index = to_datetime(index)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


In [8]:
predictions=[]
history=train.values[-results.k_ar:]

for i in range(len(test)):
    prediction=results.forecast(y=history,steps=1)
    predictions.append(prediction)
    history=np.vstack([history[1:],test.values[i]])

df_pred=pd.DataFrame(np.vstack(predictions),index=test.index,columns=test.columns)
df_pred=df_pred.clip(0.0,1.0)

df_pred.to_csv('predictions.csv')
test.to_csv('test_data.csv')